# NeuroSleep — Notebook 01: Exploratory Data Analysis

**Project**: Automated sleep stage classification from EEG signals using deep learning.

**Author**: Marcos Vinícius Rocha Gomes

**Date**: May 2026

---

## Objectives of this notebook

1. Download a sample of the Sleep-EDF Expanded dataset (PhysioNet).
2. Inspect the raw EEG recordings and understand the data structure.
3. Visualize raw signals across multiple channels.
4. Load and parse the hypnogram (sleep stage annotations).
5. Segment the recording into 30-second epochs (the standard for sleep scoring).
6. Analyze the distribution of sleep stages and identify class imbalance.
7. Define next steps for the preprocessing pipeline.

## Dataset background

The Sleep-EDF Expanded dataset contains overnight polysomnographic (PSG) recordings from healthy subjects, with sleep stages manually annotated by experts according to the Rechtschaffen and Kales (R&K) standard. We will use this dataset to train models that classify 30-second epochs into one of 5 sleep stages:

- **W**: Wake
- **N1**: NREM stage 1 (light sleep, transition)
- **N2**: NREM stage 2 (light sleep)
- **N3**: NREM stage 3 (deep / slow-wave sleep)
- **REM**: Rapid Eye Movement sleep

## 1. Imports and configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
from mne.datasets.sleep_physionet.age import fetch_data

# Quieter MNE logs and warnings for a cleaner notebook output
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore")

# Plot styling
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

print(f"MNE version: {mne.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Download a sample recording

We will download the data for a single subject to keep this exploration fast. MNE caches the files locally, so subsequent runs will not re-download. The `fetch_data` function returns a list of (PSG, hypnogram) file pairs.

Each subject in Sleep-EDF has two nights of recording; we will work with the first night.

In [ ]:
SUBJECT_ID = 0  # First subject in the dataset
RECORDING_NIGHT = 1  # First night

files = fetch_data(subjects=[SUBJECT_ID], recording=[RECORDING_NIGHT])
psg_file, hypnogram_file = files[0]

print(f"PSG file: {Path(psg_file).name}")
print(f"Hypnogram file: {Path(hypnogram_file).name}")

## 3. Load the raw EEG recording

The PSG file is in EDF (European Data Format) — a standard for biomedical signal storage. MNE's `read_raw_edf` loads it as a `Raw` object.

In [ ]:
raw = mne.io.read_raw_edf(psg_file, preload=True, stim_channel="Event marker")

print("=== Recording metadata ===")
print(f"Sampling frequency: {raw.info['sfreq']} Hz")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Recording duration: {raw.times[-1] / 3600:.2f} hours")
print(f"Number of samples: {raw.n_times:,}")
print(f"\nChannel names: {raw.ch_names}")

### Observations

- The recording is at **100 Hz** sampling rate, which is standard for Sleep-EDF.
- It contains multiple channels: 2 EEG (Fpz-Cz, Pz-Oz), 1 EOG (eye movement), 1 EMG (muscle activity), respiration, body temperature, etc.
- For sleep staging, the most informative channels are the **EEG and EOG** channels.

## 4. Visualize the raw signals

Let's plot a 30-second window of the EEG and EOG channels to see what the data actually looks like.

In [ ]:
# Select the most relevant channels for sleep staging
channels_to_plot = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]

# Pick a 30-second window from the middle of the recording
start_time = raw.times[-1] / 2  # halfway through the night
duration = 30  # seconds (one epoch)

sfreq = int(raw.info["sfreq"])
start_sample = int(start_time * sfreq)
end_sample = start_sample + duration * sfreq

fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(14, 8), sharex=True)

for ax, ch_name in zip(axes, channels_to_plot):
    ch_idx = raw.ch_names.index(ch_name)
    signal = raw.get_data()[ch_idx, start_sample:end_sample]
    time_axis = np.arange(len(signal)) / sfreq
    ax.plot(time_axis, signal * 1e6, linewidth=0.8)  # convert V -> µV for readability
    ax.set_ylabel(f"{ch_name}\n(µV)")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (seconds)")
fig.suptitle("Raw biosignals — 30-second window", fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

## 5. Load the hypnogram (sleep stage annotations)

The hypnogram file is also in EDF+ format and contains annotations marking the sleep stage of each 30-second segment. We map them to the standard 5-class scheme.

In [ ]:
annotations = mne.read_annotations(hypnogram_file)

# Mapping from raw annotation strings to standard sleep stages.
# In the R&K scheme, Stage 4 was merged into Stage 3 to form N3.
STAGE_MAPPING = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
}

# Build a DataFrame for easier inspection
annotations_df = pd.DataFrame({
    "onset_sec": annotations.onset,
    "duration_sec": annotations.duration,
    "raw_label": annotations.description,
})
annotations_df["stage"] = annotations_df["raw_label"].map(STAGE_MAPPING)

print("First 10 annotations:")
annotations_df.head(10)

Notice that each annotation has a duration: an annotation marks a continuous block where the subject was in the same stage. We will need to **expand** these into one label per 30-second epoch in the next step.

## 6. Segment the recording into 30-second epochs

Sleep scoring follows the AASM standard: the entire recording is split into non-overlapping 30-second windows (epochs), and each one receives a single stage label.

We will use MNE's `Epochs` machinery via `events_from_annotations`.

In [ ]:
# Restrict to known stages (drop "unknown" / movement / etc.)
EVENT_ID = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,  # merged with N3
    "Sleep stage R": 5,
}

# Attach annotations to the Raw object
raw.set_annotations(annotations, emit_warning=False)

# Convert annotations into MNE events.
# chunk_duration=30 ensures we get one event every 30 seconds.
events, event_id_used = mne.events_from_annotations(
    raw, event_id=EVENT_ID, chunk_duration=30.0
)

print(f"Total number of 30-s epochs found: {len(events)}")
print(f"Event ID mapping used: {event_id_used}")

In [ ]:
# Build Epochs object (epoch = 30 s, starting at the event)
tmax = 30.0 - 1.0 / raw.info["sfreq"]
epochs = mne.Epochs(
    raw=raw,
    events=events,
    event_id=event_id_used,
    tmin=0.0,
    tmax=tmax,
    baseline=None,
    preload=True,
)

print(epochs)
print(f"\nShape of epochs data: {epochs.get_data().shape}")
print("  -> (n_epochs, n_channels, n_samples_per_epoch)")

## 7. Sleep stage distribution

Class imbalance is a well-known issue in sleep staging: N2 dominates (~50% of sleep), while N1 and REM are minority classes. Let's verify it on this subject.

In [ ]:
# Map integer event codes back to readable stage names
CODE_TO_STAGE = {1: "W", 2: "N1", 3: "N2", 4: "N3", 5: "REM"}
STAGE_ORDER = ["W", "N1", "N2", "N3", "REM"]

epoch_labels = pd.Series([CODE_TO_STAGE[c] for c in events[:, 2]], name="stage")
stage_counts = epoch_labels.value_counts().reindex(STAGE_ORDER, fill_value=0)
stage_props = (stage_counts / stage_counts.sum() * 100).round(2)

summary = pd.DataFrame({"count": stage_counts, "percentage": stage_props})
print("Sleep stage distribution for this subject:")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    x=stage_counts.index,
    y=stage_counts.values,
    order=STAGE_ORDER,
    palette="viridis",
    ax=ax,
)
ax.set_title(f"Sleep stage distribution — subject {SUBJECT_ID}, night {RECORDING_NIGHT}")
ax.set_xlabel("Sleep stage")
ax.set_ylabel("Number of 30-s epochs")
for i, v in enumerate(stage_counts.values):
    ax.text(i, v + 5, f"{stage_props.iloc[i]:.1f}%", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 8. Hypnogram visualization

A hypnogram is the canonical way to visualize a full night of sleep — stages plotted over time. This gives us intuition for sleep architecture.

In [ ]:
# Order stages so the plot reads naturally: Wake at top, deep sleep at bottom
STAGE_TO_Y = {"W": 5, "REM": 4, "N1": 3, "N2": 2, "N3": 1}
y_to_label = {v: k for k, v in STAGE_TO_Y.items()}

time_hours = events[:, 0] / raw.info["sfreq"] / 3600
y_values = [STAGE_TO_Y[CODE_TO_STAGE[c]] for c in events[:, 2]]

fig, ax = plt.subplots(figsize=(14, 4))
ax.step(time_hours, y_values, where="post", linewidth=1.2, color="#2c3e50")
ax.set_yticks(list(y_to_label.keys()))
ax.set_yticklabels(list(y_to_label.values()))
ax.set_xlabel("Time (hours from recording start)")
ax.set_ylabel("Sleep stage")
ax.set_title(f"Hypnogram — subject {SUBJECT_ID}, night {RECORDING_NIGHT}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### What to look for in the hypnogram

- **Sleep cycles** of ~90 minutes are typical.
- **N3 (deep sleep)** dominates the first half of the night.
- **REM episodes** lengthen toward morning.
- Long stretches of Wake before sleep onset and after final awakening are normal.

## 9. Example signal per sleep stage

Different stages have characteristic EEG signatures. Let's plot one example 30-s epoch from each stage.

In [ ]:
fig, axes = plt.subplots(len(STAGE_ORDER), 1, figsize=(14, 10), sharex=True)

ch_name = "EEG Fpz-Cz"
ch_idx = epochs.ch_names.index(ch_name)

for ax, stage in zip(axes, STAGE_ORDER):
    # Find the first epoch of this stage
    stage_mask = epoch_labels == stage
    if not stage_mask.any():
        ax.set_title(f"{stage}: no epochs available")
        continue
    first_idx = int(np.where(stage_mask)[0][0])
    signal = epochs.get_data()[first_idx, ch_idx, :] * 1e6  # µV
    time_axis = np.arange(len(signal)) / raw.info["sfreq"]
    ax.plot(time_axis, signal, linewidth=0.7, color="#34495e")
    ax.set_ylabel(f"{stage}\n(µV)")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (seconds)")
fig.suptitle(f"Example epoch per sleep stage — channel {ch_name}", fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

### Visual differences to look for

- **W (Wake)**: low-amplitude, high-frequency activity, often with eye-blink artifacts.
- **N1**: low-amplitude mixed frequencies, transitional.
- **N2**: presence of sleep spindles (12–14 Hz bursts) and K-complexes.
- **N3**: high-amplitude slow waves (delta rhythm, 0.5–4 Hz).
- **REM**: similar to N1 in EEG, but with characteristic rapid eye movements visible on the EOG channel.

## 10. Signal-level statistics by stage

A first quantitative hint that EEG amplitude varies with stage: N3 (slow-wave sleep) should show the highest amplitudes.

In [ ]:
data = epochs.get_data()[:, ch_idx, :] * 1e6  # all epochs of the chosen channel, in µV

stats_df = pd.DataFrame({
    "stage": epoch_labels.values,
    "mean": data.mean(axis=1),
    "std": data.std(axis=1),
    "max_abs": np.max(np.abs(data), axis=1),
})

summary_by_stage = stats_df.groupby("stage").agg(
    n_epochs=("std", "count"),
    mean_amp=("mean", "mean"),
    avg_std=("std", "mean"),
    avg_max_abs=("max_abs", "mean"),
).reindex(STAGE_ORDER)

print("Signal statistics by stage (channel EEG Fpz-Cz, µV):")
summary_by_stage.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=stats_df,
    x="stage",
    y="std",
    order=STAGE_ORDER,
    palette="viridis",
    showfliers=False,
    ax=ax,
)
ax.set_title("EEG amplitude (std per epoch) by sleep stage")
ax.set_xlabel("Sleep stage")
ax.set_ylabel("Std of signal within epoch (µV)")
plt.tight_layout()
plt.show()

**Interpretation**: if the boxplot shows visibly higher amplitudes (std) for N3, this confirms that simple amplitude-based features already carry some discriminative power — a useful sanity check before moving to more elaborate features and deep learning.

## 11. Summary and next steps

### What we learned in this notebook

- Sleep-EDF provides high-quality polysomnographic recordings, sampled at 100 Hz, with expert-annotated hypnograms.
- The standard analysis unit is the **30-second epoch**, labeled with one of 5 sleep stages.
- The dataset is **class-imbalanced**: N2 dominates, and N1 / REM are minority classes — we will need to handle this during training (class weights, stratified sampling, or focal loss).
- EEG signals show clear visual differences across stages, with N3 having the most prominent slow-wave amplitude.

### Next steps (Notebook 02)

1. **Preprocessing pipeline**: bandpass filter (0.3–35 Hz), notch filter, robust normalization per recording.
2. **Feature engineering**: time-domain (Hjorth parameters, RMS, zero-crossings), frequency-domain (relative power per band: δ, θ, α, β, γ), and wavelet features.
3. **Multi-subject loading**: extend the pipeline to load all subjects and save processed features to disk.
4. **Baseline modeling**: Random Forest and SVM on engineered features (Notebook 03).
5. **Deep learning**: CNN 1D on raw signal in TensorFlow/Keras (Notebook 04).